In [ ]:
import sys
import importlib
import pandas as pd
import re
sys.path.append("../src")

import pipeline
importlib.reload(pipeline)

In [ ]:
manifest = pipeline.build_manifest("../data/raw", "../data/raw_manifest.json")

In [ ]:
for name, info in manifest.items():
    print(name, info["bytes"], info["sha256"][:12])

In [ ]:
pipeline.verify_manifest("../data/raw", "../data/raw_manifest.json")
print("raw files unchanged")

stage1 = pipeline.build_stage1("../data/raw", "../data/processed/stage1_merged.csv")

print(len(stage1))
print(stage1.groupby(["source_dataset", "source_split", "coarse_label"]).size())
print(stage1.drop(columns="text").head(3))

In [ ]:
s1 = pd.read_csv("../data/processed/stage1_merged.csv", dtype=str, keep_default_na=False)

print("total rows:", len(s1))
print("empty text:", (s1["text"].str.strip() == "").sum())
print("under 5 characters:", (s1["text"].str.len() < 5).sum())
print("over 5000 characters:", (s1["text"].str.len() > 5000).sum())
print("extra copies of the same text:", s1["text"].duplicated().sum())

In [ ]:
# 1. Display short prompts
print(s1[s1["text"].str.len() < 5][
    ["source_dataset", "coarse_label", "text"]
])

# 2. Count duplicate prompts by dataset and label
duplicates = s1[s1["text"].duplicated(keep=False)]

print(duplicates.groupby(
    ["source_dataset", "coarse_label"]
).size())

# 3. Count duplicate texts with conflicting labels
conflicts = duplicates.groupby("text")["coarse_label"].nunique()

print("Conflicting labels:", (conflicts > 1).sum())

In [ ]:
c = s1[s1["text"].isin(conflicts[conflicts > 1].index)].sort_values("text")
print("Rows involved:", len(c))

display(c[[
    "source_dataset",
    "source_split",
    "coarse_label",
    "text"
]].head(60))

In [ ]:
reg = s1[s1["source_dataset"] == "itw_regular"]

words = (
    r"do anything now|ignore previous|ignore all previous|"
    r"no restrictions|unfiltered|stay in character|"
    r"developer mode|jailbreak"
)

hit = (
    reg["text"].str.contains(words, case=False, regex=True)
    | reg["text"].str.contains(r"\bDAN\b", case=True, regex=True)
)

print("Regular rows:", len(reg))
print("Suspicious matches:", hit.sum())

display(reg[hit][["text", "coarse_label"]].head(15))

In [ ]:
sample = reg[hit].sample(20, random_state=1)

for i, (_, r) in enumerate(sample.iterrows(), start=1):
    print(i, "|", repr(r["text"][:250]))
    print()

In [ ]:
s2 = pipeline.build_stage2("../data/processed/stage1_merged.csv", "../data/processed/stage2_flagged.csv")

print("rows:", len(s2))
print("short:", s2["is_short"].sum())
print("long:", s2["is_long"].sum())
print("rows in copy groups:", (s2["dup_count"] > 1).sum())
print("rows in conflict groups:", s2["label_conflict"].sum())

In [ ]:
c2 = s2[s2["label_conflict"]].copy()
c2["tag"] = c2["source_dataset"] + ":" + c2["coarse_label"]

print("conflict groups:", c2["dup_group_id"].nunique())
print()

combo = c2.groupby("dup_group_id")["tag"].agg(lambda x: " + ".join(sorted(set(x))))
print(combo.value_counts())

In [ ]:
clear = [
    "itw_jailbreak:attack + itw_regular:safe",
    "itw_jailbreak:attack + itw_regular:safe + safeguard:attack"
]

unclear_ids = combo[~combo.isin(clear)].index

pd.set_option("display.max_colwidth", 300)

display(
    c2[c2["dup_group_id"].isin(unclear_ids)]
    [["dup_group_id", "tag", "text"]]
    .drop_duplicates()
    .sort_values("dup_group_id")
)

In [ ]:
t = s2[s2["dup_group_id"] == "g001469"]["text"].iloc[0]
print(t[:1500])

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

s2 = pd.read_csv("../data/processed/stage2_flagged.csv", dtype=str, keep_default_na=False)

jb = s2[s2["source_dataset"] == "itw_jailbreak"].reset_index(drop=True)
reg = s2[s2["source_dataset"] == "itw_regular"].reset_index(drop=True)

jb_text = jb["text"].str[:3000]
reg_text = reg["text"].str[:3000]

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
vec.fit(pd.concat([jb_text, reg_text]))

jb_m = vec.transform(jb_text)
reg_m = vec.transform(reg_text)

best_sim = []
best_idx = []
for start in range(0, reg_m.shape[0], 2000):
    chunk = reg_m[start:start + 2000]
    sims = (chunk @ jb_m.T).toarray()
    best_sim.extend(sims.max(axis=1))
    best_idx.extend(sims.argmax(axis=1))

reg["max_sim"] = best_sim
reg["closest_jb_id"] = jb["id"].iloc[best_idx].values

reg[["id", "max_sim", "closest_jb_id"]].to_csv("../data/processed/regular_similarity.csv", index=False)

labels = ["below 0.5", "0.5-0.7 (low)", "0.7-0.9 (medium)", "0.9-0.99 (high)", "0.99+ (near-exact)"]
bands = pd.cut(reg["max_sim"], bins=[0, 0.5, 0.7, 0.9, 0.99, 1.01], labels=labels, include_lowest=True)
print(bands.value_counts().sort_index())

In [ ]:
reg["opening"] = reg["text"].str.lower().str.split().str.join(" ").str[:40]
jb["opening"] = jb["text"].str.lower().str.split().str.join(" ").str[:40]

cand = reg[reg["max_sim"] >= 0.5]

fam = cand.groupby("opening").size().sort_values(ascending=False)
jb_counts = jb.groupby("opening").size()

table = pd.DataFrame({"regular_rows": fam})
table["jailbreak_file_rows"] = jb_counts.reindex(table.index).fillna(0).astype(int)

sizes = cand.groupby("opening")["opening"].transform("size")
print("rows checked:", len(cand))
print("rows in families of 3 or more:", (sizes >= 3).sum(), "| other rows:", (sizes < 3).sum())

pd.set_option("display.max_colwidth", 60)
display(table.head(25))

In [ ]:
s3 = pd.read_csv("../data/processed/stage2_flagged.csv", dtype=str, keep_default_na=False)

s3["norm"] = s3["text"].str.lower().str.split().str.join(" ")

s3["final_label"] = s3["coarse_label"]
s3["label_source"] = "original"
s3["review_status"] = "unreviewed"

jb_norms = set(s3.loc[s3["source_dataset"] == "itw_jailbreak", "norm"])

rule_a = (
    (s3["source_dataset"] != "itw_jailbreak")
    & (s3["coarse_label"] == "safe")
    & (s3["norm"].isin(jb_norms))
)

s3.loc[rule_a, "final_label"] = "attack"
s3.loc[rule_a, "label_source"] = "exact_copy_of_jailbreak_text"
s3.loc[rule_a, "review_status"] = "provisional"

s3.drop(columns="norm").to_csv("../data/processed/stage3_labeled.csv", index=False)

print("rows changed:", rule_a.sum())
print(s3[rule_a].groupby(["source_dataset", "coarse_label"]).size())
print()
print(s3["final_label"].value_counts())
print()
for _, r in s3[rule_a].head(5).iterrows():
    print(r["id"], repr(r["text"][:80]))

In [ ]:
s3 = pd.read_csv("../data/processed/stage3_labeled.csv", dtype=str, keep_default_na=False)
s3["norm"] = s3["text"].str.lower().str.split().str.join(" ")

safe_ids = [
    "itw_regular-all-004624", "itw_regular-all-007930",
    "itw_regular-all-009254", "itw_regular-all-012303",
    "itw_regular-all-009714", "itw_regular-all-009716",
    "itw_regular-all-009923", "itw_regular-all-010760",
]
amb_ids = [
    "safeguard-train-002952", "itw_regular-all-000311",
    "itw_regular-all-010309", "itw_regular-all-012985",
]

safe_norms = set(s3.loc[s3["id"].isin(safe_ids), "norm"])
amb_norms = set(s3.loc[s3["id"].isin(amb_ids), "norm"])

to_safe = s3["norm"].isin(safe_norms)
to_amb = s3["norm"].isin(amb_norms)

s3.loc[to_safe, "final_label"] = "safe"
s3.loc[to_safe, "label_source"] = "claude_review"
s3.loc[to_safe, "review_status"] = "provisional"

s3.loc[to_amb, "final_label"] = "ambiguous"
s3.loc[to_amb, "label_source"] = "claude_review"
s3.loc[to_amb, "review_status"] = "provisional"

s3.drop(columns="norm").to_csv("../data/processed/stage3b_reviewed.csv", index=False)

print("set to safe:", to_safe.sum(), "| set to ambiguous:", to_amb.sum())
print(s3[to_safe | to_amb].groupby(["source_dataset", "coarse_label", "final_label"]).size())
print()
print(s3["final_label"].value_counts())

In [ ]:
s3 = pd.read_csv("../data/processed/stage3b_reviewed.csv", dtype=str, keep_default_na=False)
sim = pd.read_csv("../data/processed/regular_similarity.csv")
s3["norm"] = s3["text"].str.lower().str.split().str.join(" ")

prefixes = (
    "i want you to act as a debater",
    "i want you to act as a linux terminal",
    "i want you to act as a storyteller",
    "i want you to act as an english translator, spelling corrector and improver",
    "i want you to act as an interviewer",
)
rule_b = s3["norm"].str.startswith(prefixes) & (s3["final_label"] == "attack")

s3.loc[rule_b, "final_label"] = "safe"
s3.loc[rule_b, "label_source"] = "claude_review"
s3.loc[rule_b, "review_status"] = "provisional"

similar_ids = set(sim.loc[sim["max_sim"] >= 0.5, "id"])
rule_c = s3["id"].isin(similar_ids) & (s3["label_source"] == "original")

s3.loc[rule_c, "final_label"] = "ambiguous"
s3.loc[rule_c, "label_source"] = "similar_to_jailbreak_unreviewed"
s3.loc[rule_c, "review_status"] = "provisional"

s3.drop(columns="norm").to_csv("../data/processed/stage3c_reviewed.csv", index=False)

print("rule B (job prompts to safe):", rule_b.sum())
print("rule C (similar to a jailbreak, set aside):", rule_c.sum())
print()
print(s3[rule_b][["id", "source_dataset", "coarse_label"]])
print()
print(s3["final_label"].value_counts())

In [ ]:
s4 = pd.read_csv("../data/processed/stage3c_reviewed.csv", dtype=str, keep_default_na=False)

pattern = r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af\u0400-\u04ff\u0600-\u06ff\u0900-\u097f\u0e00-\u0e7f\u0590-\u05ff]"

head = s4["text"].str[:500]
non_latin = head.apply(lambda t: len(re.findall(pattern, t)))
share = non_latin / head.str.len().clip(lower=1)

s4["non_english"] = share > 0.2

s4.to_csv("../data/processed/stage4_nonenglish.csv", index=False)

print("flagged:", s4["non_english"].sum())
print(s4[s4["non_english"]].groupby(["source_dataset", "final_label"]).size())
print()
print("some flagged rows:")
for t in s4[s4["non_english"]]["text"].sample(8, random_state=1):
    print(repr(t[:70]))
print()
print("some NOT flagged rows that contain a few non-Latin characters:")
partial = s4[(non_latin > 0) & (~s4["non_english"])]
for t in partial["text"].sample(5, random_state=1):
    print(repr(t[:70]))

In [ ]:
is6 = pd.read_csv("../data/processed/stage4_nonenglish.csv", dtype=str, keep_default_na=False)
s6["norm"] = s6["text"].str.lower().str.split().str.join(" ")


def set_label(mask, label, source):
    s6.loc[mask, "final_label"] = label
    s6.loc[mask, "label_source"] = source
    s6.loc[mask, "review_status"] = "provisional"


def same_text_as(ids):
    texts = set(s6.loc[s6["id"].isin(ids), "norm"])
    return s6["norm"].isin(texts)


ids_to_attack = ["itw_regular-all-009889"]
ids_to_safe = ["itw_jailbreak-all-001249", "itw_jailbreak-all-000215"]
ids_to_amb = [
    "itw_regular-all-012914", "itw_regular-all-007762",
    "itw_jailbreak-all-001352", "itw_jailbreak-all-001111",
    "itw_jailbreak-all-001162", "deepset-test-000105",
]

set_label(same_text_as(ids_to_attack), "attack", "claude_review")
set_label(same_text_as(ids_to_safe), "safe", "claude_review")
set_label(same_text_as(ids_to_amb), "ambiguous", "claude_review")

is_template = (
    s6["text"].str.contains("{{user}}", regex=False)
    | s6["text"].str.contains("{{char}}", regex=False)
)
set_label(is_template, "ambiguous", "roleplay_template_rule")

is_exception = s6["norm"].str.contains("exception to ai usual ethical protocols", regex=False)
set_label(is_exception, "attack", "claude_review_family_rule")

s6.drop(columns="norm").to_csv("../data/processed/stage6_audited.csv", index=False)

print("template rows:", is_template.sum())
print("exception-family rows:", is_exception.sum())
print()
print(s6["label_source"].value_counts())
print()
print(s6.groupby(["source_dataset", "final_label"]).size())
print()
print(s6["final_label"].value_counts())

In [ ]:
d = pd.read_csv("../data/processed/stage6_audited.csv", dtype=str, keep_default_na=False)
d["non_english"] = d["non_english"] == "True"

keep = d[d["final_label"].isin(["safe", "attack"]) & (~d["non_english"])].copy()
keep["group"] = keep["text"].str.lower().str.split().str.join(" ").str[:40]
keep["is_attack"] = keep["final_label"] == "attack"

table = keep.groupby("group").agg(rows=("id", "size"), attack_rows=("is_attack", "sum"))
table = table.sort_values("rows", ascending=False)

print("rows kept:", len(keep), "| groups:", len(table))
print(keep["final_label"].value_counts())
print()
print("rows in the 10 biggest groups:", table["rows"].head(10).sum())

pd.set_option("display.max_colwidth", 60)
display(table.head(15))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

d = pd.read_csv("../data/processed/stage6_audited.csv", dtype=str, keep_default_na=False)
d["non_english"] = d["non_english"] == "True"

keep = d[d["final_label"].isin(["safe", "attack"]) & (~d["non_english"])].copy()
keep = keep.reset_index(drop=True)
keep["group"] = keep["text"].str.lower().str.split().str.join(" ").str[:40]

group_size = keep.groupby("group")["id"].transform("size")
giant = group_size >= 100
keep["split"] = "train"

rest = keep[~giant]
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=42)
for fold, (_, idx) in enumerate(sgkf.split(rest, rest["final_label"], rest["group"])):
    if fold == 0:
        keep.loc[rest.index[idx], "split"] = "test"
    elif fold == 1:
        keep.loc[rest.index[idx], "split"] = "val"

keep.to_csv("../data/processed/dataset_v1.csv", index=False)

print("giant groups:", keep.loc[giant, "group"].nunique(), "| rows:", giant.sum())
print("groups appearing in more than one split:", (keep.groupby("group")["split"].nunique() > 1).sum())
print()
print(pd.crosstab(keep["split"], keep["final_label"]))
print()
print(pd.crosstab(keep["split"], keep["source_dataset"]))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv("../data/processed/dataset_v2.csv", dtype=str, keep_default_na=False)
df["y"] = (df["final_label"] == "attack").astype(int)
df["short_text"] = df["text"].str[:3000]

train = df[df["split"] == "train"]
val = df[df["split"] == "val"].copy()

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=200000)
X_train = vec.fit_transform(train["short_text"])
X_val = vec.transform(val["short_text"])

model = LogisticRegression(max_iter=1000,random_state=42)
model.fit(X_train, train["y"])

val["pred"] = model.predict(X_val)

print(classification_report(val["y"], val["pred"], target_names=["safe", "attack"], digits=3))
print(confusion_matrix(val["y"], val["pred"]))
print()

for src, g in val.groupby("source_dataset"):
    attacks = g[g["y"] == 1]
    safes = g[g["y"] == 0]
    caught = (attacks["pred"] == 1).mean() if len(attacks) else float("nan")
    flagged = (safes["pred"] == 1).mean() if len(safes) else float("nan")
    print(src, "| attack rows:", len(attacks), "| caught:", round(caught, 3),
          "| safe rows:", len(safes), "| wrongly flagged:", round(flagged, 3))

In [ ]:
df = pd.read_csv("../data/processed/dataset_v1.csv", dtype=str, keep_default_na=False)

german = re.compile(r"\b(der|die|das|und|ist|nicht|ich|ein|eine|mit|für|auf|zu|wie|was|sich|von|den|dem|im)\b")

head = df["text"].str.lower().str[:400]
df["german_hits"] = head.apply(lambda t: len(german.findall(t)))
df["is_german"] = df["german_hits"] >= 3

print("flagged as German:", df["is_german"].sum())
print(pd.crosstab(df["source_dataset"], [df["is_german"], df["final_label"]]))
print()
for t in df[df["is_german"]]["text"].sample(8, random_state=1):
    print(repr(t[:90]))
print()
print("rows with 1-2 hits (not flagged), a few examples:")
partial = df[(df["german_hits"] >= 1) & (~df["is_german"])]
for t in partial["text"].sample(6, random_state=1):
    print(repr(t[:90]))

In [ ]:
df = pd.read_csv("../data/processed/dataset_v1.csv", dtype=str, keep_default_na=False)

german = re.compile(r"\b(der|die|das|und|ist|nicht|ich|ein|eine|mit|für|auf|zu|wie|sich|von|den|dem|im|sie|sind)\b")

head = df["text"].str.lower().str[:400]
df["german_hits"] = head.apply(lambda t: len(german.findall(t)))
df["is_german"] = df["german_hits"] >= 3

print("flagged as German:", df["is_german"].sum())
print(pd.crosstab(df["source_dataset"], [df["is_german"], df["final_label"]]))
print()
for t in df[df["is_german"]]["text"].sample(10, random_state=3):
    print(repr(t[:90]))

english = df[~df["is_german"]].drop(columns=["german_hits", "is_german"])
english.to_csv("../data/processed/dataset_v2.csv", index=False)

print()
print("rows before:", len(df), "| after:", len(english))
print(pd.crosstab(english["split"], english["final_label"]))